# Export Dashboard Analytics Snapshot

Export curated Gold analytics data to a portable Parquet
snapshot for the Streamlit dashboard.

The snapshot enables local and public dashboard execution
without Azure credentials or active Databricks compute.

In [0]:
from datetime import datetime, timezone
from pathlib import Path
import json

SOURCE_VIEW = "f1_lakehouse.analytics.vw_race_results"

EXPECTED_SEASONS = set(range(2021, 2026))

BUSINESS_KEY = ["season", "round", "driver_id"]

repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

assert (repo_root / "dashboard").is_dir(), (
    "Dashboard directory not found in repository"
)

EXPORT_DIR = repo_root / "dashboard" / "data"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source: {SOURCE_VIEW}")
print(f"Export directory: {EXPORT_DIR}")

In [0]:
df_analytics = spark.table(SOURCE_VIEW)

df_analytics.printSchema()

print(f"Records: {df_analytics.count()}")
print(f"Columns: {len(df_analytics.columns)}")

display(df_analytics.limit(10))

In [0]:
from pyspark.sql import functions as F

record_count = df_analytics.count()

assert record_count > 0, "Empty analytics dataset"

actual_seasons = {
    row["season"]
    for row in df_analytics.select("season").distinct().collect()
}

assert actual_seasons == EXPECTED_SEASONS, (
    "Unexpected historical season coverage"
)

duplicate_keys = (
    df_analytics
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_keys == 0, "Duplicate business keys"

invalid_points = df_analytics.filter(
    F.col("points").isNull() | (F.col("points") < 0)
).count()

assert invalid_points == 0, "Invalid points detected"

print(f"Records: {record_count}")
print(f"Seasons: {sorted(actual_seasons)}")
print(f"Duplicate keys: {duplicate_keys}")
print(f"Invalid points: {invalid_points}")

print("Snapshot source validation: PASSED")

In [0]:
import pandas as pd

rows = (
    df_analytics
    .orderBy("season", "round", "driver_id")
    .collect()
)

print(f"Collected records: {len(rows)}")

df_snapshot = pd.DataFrame(
    [row.asDict(recursive=True) for row in rows],
    columns=df_analytics.columns,
)

print(f"Pandas DataFrame created: {df_snapshot.shape}")

snapshot_path = EXPORT_DIR / "race_results.parquet"

df_snapshot.to_parquet(
    snapshot_path,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Exported records: {len(df_snapshot)}")
print(f"File: {snapshot_path.name}")
print(f"Size: {snapshot_path.stat().st_size / 1024:.2f} KB")

In [0]:
metadata = {
    "dataset": "F1 Race Results",
    "source": SOURCE_VIEW,
    "source_api": "Jolpica F1 API",
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "season_from": int(df_snapshot["season"].min()),
    "season_to": int(df_snapshot["season"].max()),
    "record_count": len(df_snapshot),
    "column_count": len(df_snapshot.columns),
    "format": "parquet",
    "compression": "snappy",
    "file_name": snapshot_path.name,
    "file_size_bytes": snapshot_path.stat().st_size,
}

metadata_path = EXPORT_DIR / "metadata.json"

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print(json.dumps(metadata, indent=2))

In [0]:
import pandas as pd

df_exported = pd.read_parquet(snapshot_path)

assert len(df_exported) == record_count

assert list(df_exported.columns) == list(df_snapshot.columns)

assert not df_exported.duplicated(BUSINESS_KEY).any()

pd.testing.assert_frame_equal(
    df_snapshot.reset_index(drop=True),
    df_exported.reset_index(drop=True),
    check_dtype=False,
)

print("Parquet read-back validation: PASSED")

print(df_exported.dtypes)

display(df_exported.head(10))

In [0]:
for file in sorted(EXPORT_DIR.iterdir()):
    if file.is_file():
        print(f"{file.name}: {file.stat().st_size / 1024:.2f} KB")